In [1]:
!pip install pandas numpy scikit-learn xgboost matplotlib seaborn flask flask-cors joblib requests

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, roc_auc_score, roc_curve)
from xgboost import XGBClassifier
import joblib
import warnings
warnings.filterwarnings('ignore')
print("✅ All libraries imported!")

✅ All libraries imported!


In [3]:
import os
os.chdir(r'C:\Users\adhil\OneDrive\Desktop\Phishing Project')

df = pd.read_csv('phishing.csv')
df.columns = df.columns.str.strip()

print("Columns:", df.columns.tolist())
print("Shape:", df.shape)

# Find and rename label column automatically
for col in ['Result','result','class','Class','phishing','label','target']:
    if col in df.columns:
        df = df.rename(columns={col: 'Result'})
        print(f"✅ Label column found: '{col}' → renamed to 'Result'")
        break

print("\nClass distribution:")
print(df['Result'].value_counts())

Columns: ['Index', 'UsingIP', 'LongURL', 'ShortURL', 'Symbol@', 'Redirecting//', 'PrefixSuffix-', 'SubDomains', 'HTTPS', 'DomainRegLen', 'Favicon', 'NonStdPort', 'HTTPSDomainURL', 'RequestURL', 'AnchorURL', 'LinksInScriptTags', 'ServerFormHandler', 'InfoEmail', 'AbnormalURL', 'WebsiteForwarding', 'StatusBarCust', 'DisableRightClick', 'UsingPopupWindow', 'IframeRedirection', 'AgeofDomain', 'DNSRecording', 'WebsiteTraffic', 'PageRank', 'GoogleIndex', 'LinksPointingToPage', 'StatsReport', 'class']
Shape: (11054, 32)
✅ Label column found: 'class' → renamed to 'Result'

Class distribution:
Result
 1    6157
-1    4897
Name: count, dtype: int64


In [4]:
# Drop label and any ID columns
drop_cols = ['Result']
if 'Index' in df.columns:
    drop_cols.append('Index')

X = df.drop(drop_cols, axis=1)
y = df['Result'].map({-1: 0, 1: 1})

print("Features:", X.columns.tolist())
print("Feature count:", X.shape[1])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"\n✅ Train: {len(X_train)} | Test: {len(X_test)}")

# Save feature names for API use later
feature_names = X.columns.tolist()
print("\nFeature names saved!")

Features: ['UsingIP', 'LongURL', 'ShortURL', 'Symbol@', 'Redirecting//', 'PrefixSuffix-', 'SubDomains', 'HTTPS', 'DomainRegLen', 'Favicon', 'NonStdPort', 'HTTPSDomainURL', 'RequestURL', 'AnchorURL', 'LinksInScriptTags', 'ServerFormHandler', 'InfoEmail', 'AbnormalURL', 'WebsiteForwarding', 'StatusBarCust', 'DisableRightClick', 'UsingPopupWindow', 'IframeRedirection', 'AgeofDomain', 'DNSRecording', 'WebsiteTraffic', 'PageRank', 'GoogleIndex', 'LinksPointingToPage', 'StatsReport']
Feature count: 30

✅ Train: 8843 | Test: 2211

Feature names saved!


In [5]:
models = {
    'Logistic Regression':  LogisticRegression(max_iter=1000),
    'Random Forest':        RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost':              XGBClassifier(eval_metric='logloss', random_state=42),
    'Gradient Boosting':    GradientBoostingClassifier(n_estimators=100, random_state=42),
    'Neural Network (MLP)': MLPClassifier(hidden_layer_sizes=(64,32), max_iter=500, random_state=42),
}

results = {}

for name, model in models.items():
    if name in ['Logistic Regression', 'Neural Network (MLP)']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

    results[name] = {
        'model':     model,
        'y_pred':    y_pred,
        'y_prob':    y_prob,
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall':    recall_score(y_test, y_pred),
        'f1':        f1_score(y_test, y_pred),
        'roc_auc':   roc_auc_score(y_test, y_prob),
    }
    print(f"✅ {name} → Accuracy: {results[name]['accuracy']:.4f}")

✅ Logistic Regression → Accuracy: 0.9389
✅ Random Forest → Accuracy: 0.9738
✅ XGBoost → Accuracy: 0.9706
✅ Gradient Boosting → Accuracy: 0.9539
✅ Neural Network (MLP) → Accuracy: 0.9720


In [6]:
# Find best model
best_model_name = max(results, key=lambda n: results[n]['f1'])
best_model      = results[best_model_name]['model']

print(f"🏆 Best model: {best_model_name}")

# Save model and scaler
joblib.dump(best_model, 'phishing_model.pkl')
joblib.dump(scaler,     'scaler.pkl')

# Save feature names so API uses correct columns
import json
with open('feature_names.json', 'w') as f:
    json.dump(feature_names, f)

print("✅ phishing_model.pkl saved!")
print("✅ scaler.pkl saved!")
print("✅ feature_names.json saved!")
print("\nFeature names:", feature_names)

🏆 Best model: Random Forest
✅ phishing_model.pkl saved!
✅ scaler.pkl saved!
✅ feature_names.json saved!

Feature names: ['UsingIP', 'LongURL', 'ShortURL', 'Symbol@', 'Redirecting//', 'PrefixSuffix-', 'SubDomains', 'HTTPS', 'DomainRegLen', 'Favicon', 'NonStdPort', 'HTTPSDomainURL', 'RequestURL', 'AnchorURL', 'LinksInScriptTags', 'ServerFormHandler', 'InfoEmail', 'AbnormalURL', 'WebsiteForwarding', 'StatusBarCust', 'DisableRightClick', 'UsingPopupWindow', 'IframeRedirection', 'AgeofDomain', 'DNSRecording', 'WebsiteTraffic', 'PageRank', 'GoogleIndex', 'LinksPointingToPage', 'StatsReport']


In [7]:
import os
files = os.listdir(r'C:\Users\adhil\OneDrive\Desktop\Phishing Project')
print("Files in project folder:")
for f in files:
    print(" ✅", f)

Files in project folder:
 ✅ .ipynb_checkpoints
 ✅ feature_names.json
 ✅ phishing.csv
 ✅ phishing_api
 ✅ phishing_detection.ipynb
 ✅ phishing_extension
 ✅ phishing_model.pkl
 ✅ scaler.pkl
